# ClarifyMed — Diabetic Retinopathy MobileNetV2 / EfficientNet Training
### Dataset: APTOS 2019 Blindness Detection

This standalone Google Colab notebook trains and evaluates a deep convolutional neural network on the APTOS 2019 dataset to screen and grade Diabetic Retinopathy across the 5 standard ICDR stages:
- **0**: No DR
- **1**: Mild NPDR
- **2**: Moderate NPDR
- **3**: Severe NPDR
- **4**: Proliferative DR

**Hardware Recommendation:** In Colab, select **Runtime > Change runtime type > T4 GPU** (free tier).

## 1. Setup & GPU Check

In [ ]:
import torch
print("PyTorch version:", torch.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training on device:", device)
if device.type == "cuda":
    print("GPU Device:", torch.cuda.get_device_name(0))

## 2. Upload / Download Dataset
You can either upload your `aptos2019-blindness-detection.zip` file directly to the Colab files panel on the left, or use the file uploader below.

In [ ]:
import os
import zipfile
from google.colab import files

# Check if zip already exists or prompt to upload
zip_candidates = [f for f in os.listdir('.') if f.endswith('.zip') and 'aptos' in f.lower()]
if not zip_candidates:
    print("Please upload your APTOS 2019 zip file:")
    uploaded = files.upload()
    zip_name = list(uploaded.keys())[0]
else:
    zip_name = zip_candidates[0]
    print(f"Found dataset zip: {zip_name}")

# Extract dataset
extract_dir = "aptos_data"
os.makedirs(extract_dir, exist_ok=True)
print(f"Extracting {zip_name}...")
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)
print("Extraction complete!")
print("Contents:", os.listdir(extract_dir))

## 3. Dataset Validation & Split

In [ ]:
import csv
import random
from pathlib import Path
from collections import Counter

data_path = Path("aptos_data")
csv_candidates = list(data_path.glob("*.csv")) + list(Path(".").glob("*.csv"))
csv_file = [c for c in csv_candidates if "train" in c.name.lower() or "aptos" in c.name.lower()][0]
print("Using annotations CSV:", csv_file)

# Find images folder
images_dir = data_path / "train_images"
if not images_dir.is_dir():
    images_dir = data_path
print("Using images directory:", images_dir)

records = []
with open(csv_file, encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)
    for row in reader:
        id_code = row.get("id_code", "").strip()
        diagnosis = int(row.get("diagnosis", 0))
        for ext in [".png", ".jpg", ".jpeg"]:
            p = images_dir / f"{id_code}{ext}"
            if p.is_file():
                records.append({"path": p, "label": diagnosis})
                break

print(f"Successfully matched {len(records)} images with labels.")
counts = Counter(r['label'] for r in records)
print("Class distribution:", dict(sorted(counts.items())))

# Stratified 70% Train, 15% Dev, 15% Test split
random.seed(42)
random.shuffle(records)
n = len(records)
n_train = int(n * 0.70)
n_dev = int(n * 0.15)

train_data = records[:n_train]
dev_data = records[n_train:n_train + n_dev]
test_data = records[n_train + n_dev:]
print(f"Splits -> Train: {len(train_data)} | Dev: {len(dev_data)} | Test: {len(test_data)}")

## 4. PyTorch Model Architecture (MobileNetV2 / EfficientNet)
We use a pre-trained ImageNet backbone fine-tuned for 5-class multi-class cross-entropy classification.

In [ ]:
import torch.nn as nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image

class APTOSDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        img = Image.open(item["path"]).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, item["label"]

train_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

eval_tf = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_loader = DataLoader(APTOSDataset(train_data, train_tf), batch_size=32, shuffle=True, num_workers=2)
dev_loader = DataLoader(APTOSDataset(dev_data, eval_tf), batch_size=32, num_workers=2)
test_loader = DataLoader(APTOSDataset(test_data, eval_tf), batch_size=32, num_workers=2)

# Build Model
model = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 5)
model = model.to(device)
print("Model loaded and classifier adapted for 5 classes.")

## 5. Training Loop with Evaluation

In [ ]:
import numpy as np
import time

EPOCHS = 8
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_dev_acc = 0.0
best_model_weights = None

for epoch in range(1, EPOCHS + 1):
    start_time = time.time()
    model.train()
    running_loss = 0.0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        preds = model(imgs)
        loss = criterion(preds, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * imgs.size(0)
    scheduler.step()

    train_loss = running_loss / len(train_data)

    # Evaluate on Dev set
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for imgs, labels in dev_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            preds = torch.argmax(model(imgs), dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    dev_acc = correct / total
    elapsed = time.time() - start_time

    print(f"Epoch [{epoch}/{EPOCHS}] - Loss: {train_loss:.4f} | Dev Acc: {dev_acc:.2%} | Time: {elapsed:.1f}s")
    if dev_acc > best_dev_acc:
        best_dev_acc = dev_acc
        best_model_weights = model.state_dict().copy()

print(f"Best Dev Accuracy: {best_dev_acc:.2%}")

## 6. Final Test Set Evaluation & Metrics Export

In [ ]:
import json
from datetime import datetime, timezone
from sklearn.metrics import classification_report, confusion_matrix, cohen_kappa_score

# Load best weights
if best_model_weights is not None:
    model.load_state_dict(best_model_weights)

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        imgs = imgs.to(device)
        preds = torch.argmax(model(imgs), dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

test_acc = float(np.mean(all_preds == all_labels))
kappa = float(cohen_kappa_score(all_labels, all_preds, weights="quadratic"))
print(f"Final Test Accuracy: {test_acc:.2%}")
print(f"Quadratic Weighted Kappa (QWK): {kappa:.4f}")

classes = ["No DR", "Mild NPDR", "Moderate NPDR", "Severe NPDR", "Proliferative DR"]
report_dict = classification_report(all_labels, all_preds, target_names=classes, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=classes))

# Save evaluation.json
eval_data = {
    "architecture": "MobileNetV2",
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "dataset": "APTOS 2019 Blindness Detection",
    "test_accuracy": test_acc,
    "quadratic_weighted_kappa": kappa,
    "classification_report": report_dict,
    "confusion_matrix": confusion_matrix(all_labels, all_preds).tolist(),
    "classes": classes,
    "sample_counts": {"train": len(train_data), "dev": len(dev_data), "test": len(test_data)}
}

with open("evaluation.json", "w", encoding="utf-8") as f:
    json.dump(eval_data, f, indent=2)

# Save model.pt
torch.save(model, "model.pt")
print("Saved model.pt and evaluation.json successfully!")

## 7. Download Trained Artifacts to your Laptop
Run the cell below to download `model.pt` and `evaluation.json`. 
Then place them into your local ClarifyMed folder at:
`ClarifyMed/ml/artifacts/retinopathy-efficientnet/`

In [ ]:
from google.colab import files
print("Downloading model.pt...")
files.download("model.pt")
print("Downloading evaluation.json...")
files.download("evaluation.json")